# Sup Figure 2 | GC Content, RBS, and CAI vs. Fitness Magnitude of Positive Hits

## Configuration

In [ ]:
from pathlib import Path

# --- data directory (populate yourself -- see README's Data section) ---
DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- paths ---
GENE_DATA_PATH = DATA_DIR / "gene_fitness_results_with_annotations.parquet"

# --- analysis parameters ---
CONDITION = "LB_4_salt"


## Load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

df_raw = pd.read_parquet(GENE_DATA_PATH)
print(f"{len(df_raw):,} rows loaded")


## Data preparation

In [ ]:
# --- filter to salt condition genes with a gene call ---
salt_genes = df_raw[df_raw["condition"] == CONDITION].copy()

salt_genes_filt = salt_genes[
    salt_genes["gene_call"].isin(["Hit", "Not Hit"])
].copy()

print(f"{len(salt_genes_filt):,} genes with Hit/Not Hit call")
salt_genes_filt.head()


In [ ]:
median_e_coli_gc = salt_genes[salt_genes["species_name"] == "Escherichia coli"]["gc_content"].median()

per_gene_df = (
    salt_genes_filt[["gene_id", "species_name", "gene_call", "gc_content", "CAI_rel_to_Ecoli", "Sapiens_RBS_Score", "lmer_estimate"]]
    .copy()
)
per_gene_df["is_hit"] = ((per_gene_df["gene_call"] == "Hit") & (per_gene_df["lmer_estimate"] > 0)).astype(int)
per_gene_df["gc_content_diff_ecoli"] = per_gene_df["gc_content"] - median_e_coli_gc
per_gene_df["short_name"] = per_gene_df["species_name"].apply(lambda x: f"{x.split()[0][0]}. {x.split()[1]}")

pos_hits = per_gene_df[per_gene_df["is_hit"] == 1]
pos_hits.head()


## Sup Figure 2a -- GC content vs. fitness magnitude of positive hits

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))

sns.scatterplot(data=pos_hits, x="gc_content", y="lmer_estimate", hue="short_name")
ax.set_xlabel("GC content (%)")
ax.set_ylabel("Gene fitness estimate")
plt.legend(title="Species", loc="upper right", bbox_to_anchor=(1.5, 1.0))
plt.show()
smf.ols("lmer_estimate ~ gc_content", data=pos_hits).fit().summary()

fig.savefig(RESULTS_DIR / "sup_figure2a_gc_content_vs_hit_magnitude.pdf", bbox_inches="tight")


## Sup Figure 2b -- RBS binding score vs. fitness magnitude of positive hits

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))

sns.scatterplot(data=pos_hits, x="Sapiens_RBS_Score", y="lmer_estimate", hue="short_name")
ax.set_xlabel("Predicted RBS affinity")
ax.set_ylabel("Gene fitness estimate")
plt.legend(title="Species", loc="upper right", bbox_to_anchor=(1.5, 1.0))
plt.show()
smf.ols("lmer_estimate ~ Sapiens_RBS_Score", data=pos_hits).fit().summary()

fig.savefig(RESULTS_DIR / "sup_figure2b_rbs_vs_hit_magnitude.pdf", bbox_inches="tight")


## Sup Figure 2c -- CAI relative to E. coli vs. fitness magnitude of positive hits

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))

sns.scatterplot(data=pos_hits, x="CAI_rel_to_Ecoli", y="lmer_estimate", hue="short_name")
ax.set_xlabel("CAI relative to E. coli")
ax.set_ylabel("Gene fitness estimate")
plt.legend(title="Species", loc="upper right", bbox_to_anchor=(1.5, 1.0))
plt.show()
smf.ols("lmer_estimate ~ CAI_rel_to_Ecoli", data=pos_hits).fit().summary()

fig.savefig(RESULTS_DIR / "sup_figure2c_cai_vs_hit_magnitude.pdf", bbox_inches="tight")
